# Notebook 1 — Read & Join the Tables

This notebook reads the Olist tables from the PostgreSQL database, inspects their structure, and prepares the data for joining.

The main goal is to create one machine learning table with one row per order.

## 1. Import Libraries

I use Pandas to work with the data as DataFrames. I also use SQLAlchemy and psycopg2 to connect Python to the PostgreSQL database.

In [1]:
import pandas as pd
import sqlalchemy
import psycopg2

print("Pandas version:", pd.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)
print("PostgreSQL connection library: psycopg2")

Pandas version: 3.0.5
SQLAlchemy version: 2.0.51
PostgreSQL connection library: psycopg2


In [2]:
import sys

# Show the Python version used by Jupyter
print("Python:", sys.version)

# Show the exact Python executable used by Jupyter
print("Python path:", sys.executable)

Python: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
Python path: C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe


## 3. Verify the Project Environment

I verify that the notebook is using the dedicated Qafza MLOps environment and that the required libraries are available.

In [3]:
import sys
import pandas as pd
import sqlalchemy
import psycopg2

# Check the Python environment used by the notebook
print("Python version:", sys.version)
print("Python path:", sys.executable)

# Check the main libraries used in this project
print("Pandas version:", pd.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)
print("psycopg2: working")

Python version: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
Python path: C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe
Pandas version: 3.0.5
SQLAlchemy version: 2.0.51
psycopg2: working


## 4. Connect to the PostgreSQL Database

The Olist data is stored in a PostgreSQL database running inside Docker.

I connect to the database through the local port exposed by the Docker container. The connection will be used to read the Olist tables into Python.

In [4]:
from sqlalchemy import create_engine

# Database connection settings
username = "postgres"
password = "OlistDocker2026"
host = "localhost"
port = 5433
database = "olist_db"

# Create a connection to PostgreSQL
connection_string = (
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

engine = create_engine(connection_string)

print("Database engine created successfully.")

Database engine created successfully.


## 5. List the Database Tables

I first retrieve the table names from the PostgreSQL database. This confirms which Olist tables are available before reading their data.

In [5]:
# Read the names of all tables in the public schema
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)

print("Available tables:")
print(tables)

Available tables:
                     table_name
0                     customers
1                   geolocation
2                   order_items
3                order_payments
4                 order_reviews
5                        orders
6  product_category_translation
7                      products
8                       sellers


## 6. Inspect Table Row Counts

I check the number of rows in each table to verify that the data was loaded correctly and to understand the size of each dataset before performing any joins.

In [6]:
# Check the number of rows in each table

for table_name in tables["table_name"]:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    result = pd.read_sql(query, engine)

    print(table_name, ":", result.loc[0, "row_count"], "rows")

customers : 99441 rows
geolocation : 1000163 rows
order_items : 112650 rows
order_payments : 103886 rows
order_reviews : 99224 rows
orders : 99441 rows
product_category_translation : 71 rows
products : 32951 rows
sellers : 3095 rows


## 7. Inspect the Row Grain of the Main Tables

Before joining the tables, I inspect their keys and understand what each row represents.

This is important because some tables contain multiple rows for the same order. The final machine learning dataset should contain one row per order.

In [7]:
# Read a small sample from the orders table
orders_sample = pd.read_sql(
    "SELECT * FROM orders LIMIT 5",
    engine
)

# Display the sample vertically for easier reading
orders_sample.T

,0,1,2,3,4
order_id,e481f51cbdc54678b7cc49136f2d6af7,53cdb2fc8bc7dce0b6741e2150273451,47770eb9100c2d0c44946d9cf07ec65d,949d5b44dbf5de918fe9c16f97b45f8a,ad21c59c0840e6cb83a9ceb5573f8159
customer_id,9ef432eb6251297304e76186b10a928d,b0830fb4747a6c6d20dea0b8c802d7ef,41ce2a54c0b03bf3443c3d931a367089,f88197465ea7920adcdbec7375364d82,8ab97904e6daea8866dbdbc4fb7aad2c
order_status,delivered,delivered,delivered,delivered,delivered
order_purchase_timestamp,2017-10-02 10:56:33,2018-07-24 20:41:37,2018-08-08 08:38:49,2017-11-18 19:28:06,2018-02-13 21:18:39
order_approved_at,2017-10-02 11:07:15,2018-07-26 03:24:27,2018-08-08 08:55:23,2017-11-18 19:45:59,2018-02-13 22:20:29
order_delivered_carrier_date,2017-10-04 19:55:00,2018-07-26 14:31:00,2018-08-08 13:50:00,2017-11-22 13:39:59,2018-02-14 19:46:34
order_delivered_customer_date,2017-10-10 21:25:13,2018-08-07 15:27:45,2018-08-17 18:06:29,2017-12-02 00:28:42,2018-02-16 18:17:02
order_estimated_delivery_date,2017-10-18 00:00:00,2018-08-13 00:00:00,2018-09-04 00:00:00,2017-12-15 00:00:00,2018-02-26 00:00:00


In [8]:
# Read a small sample from the order_items table
items_sample = pd.read_sql(
    "SELECT * FROM order_items LIMIT 5",
    engine
)

# Display the sample vertically for easier reading
items_sample.T

,0,1,2,3,4
order_id,00010242fe8c5a6d1ba2dd792cb16214,00018f77f2f0320c557190d7a144bdd3,000229ec398224ef6ca0657da4fc703e,00024acbcdf0a6daa1e931b038114c75,00042b26cf59d7ce69dfabb4e55b4fd9
order_item_id,1,1,1,1,1
product_id,4244733e06e7ecb4970a6e2683c13e61,e5f2d52b802189ee658865ca93d83a8f,c777355d18b72b67abbeef9df44fd0fd,7634da152a4610f1595efa32f14722fc,ac6c3623068f30de03045865e4e10089
seller_id,48436dade18ac8b2bce089ec2a041202,dd7ddc04e1b6c2c614352b383efe2d36,5b51032eddd242adc84c38acab88f23d,9d7a1d34a5052409006425275ba1c2b4,df560393f3a51e74553ab94004ba5c87
shipping_limit_date,2017-09-19 09:45:35,2017-05-03 11:05:13,2018-01-18 14:48:30,2018-08-15 10:10:18,2017-02-13 13:57:51
price,58.9,239.9,199.0,12.99,199.9
freight_value,13.29,19.93,17.87,12.79,18.14


## 8. Check Order-Level Uniqueness

I check whether `order_id` is unique in the orders table and how many order items can belong to the same order.

This helps identify one-to-many relationships before building the final order-level dataset.

In [9]:
# Check whether order_id is unique in the orders table
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM orders;
"""

orders_check = pd.read_sql(query, engine)

print(orders_check)

   total_rows  unique_orders
0       99441          99441


In [10]:
# Check how many order items belong to each order
query = """
SELECT
    COUNT(*) AS total_items,
    COUNT(DISTINCT order_id) AS orders_with_items
FROM order_items;
"""

items_check = pd.read_sql(query, engine)

print(items_check)

   total_items  orders_with_items
0       112650              98666


## 9. Check Items per Order

I check how many items are associated with each order.

This helps confirm the one-to-many relationship and determine how the item-level data should be aggregated before creating the final order-level dataset.

In [11]:
# Count the number of items for each order
query = """
SELECT
    order_id,
    COUNT(*) AS item_count
FROM order_items
GROUP BY order_id
ORDER BY item_count DESC;
"""

items_per_order = pd.read_sql(query, engine)

# Show the orders with the highest number of items
items_per_order.head(10)

,order_id,item_count
0,8272b63d03f5f79c56e9e4120aec44ef,21
1,1b15974a0141d54e36626dca3fdc731a,20
2,ab14fdcfbe524636d65ee38360e22ce8,20
3,428a2f660dc84138d969ccd69a0ab6d5,15
4,9ef13efd6949e4573a18964dd1bbe7f5,15
5,73c8ab38f07dc94389065f7eba4f297a,14
6,9bdc4d4c71aa1de4606060929dee888c,14
7,37ee401157a3a0b28c9c6d0ed8c3b24b,13
8,2c2a19b5703863c908512d135aa6accc,12
9,3a213fcdfe7d98be74ea0dc05a8b31ae,12


In [12]:
# Show basic statistics about the number of items per order
items_per_order["item_count"].describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

## 10. Check the Payment Table Grain

I inspect the payment table to determine whether each order has one or multiple payment records.

Understanding the row grain of each table is necessary before joining the data at the order level.

In [13]:
# Check the number of payment records and unique orders
query = """
SELECT
    COUNT(*) AS total_payments,
    COUNT(DISTINCT order_id) AS orders_with_payments
FROM order_payments;
"""

payments_check = pd.read_sql(query, engine)

print(payments_check)

   total_payments  orders_with_payments
0          103886                 99440


In [14]:
# Count payment records for each order
query = """
SELECT
    order_id,
    COUNT(*) AS payment_count
FROM order_payments
GROUP BY order_id
ORDER BY payment_count DESC;
"""

payments_per_order = pd.read_sql(query, engine)

# Show orders with the highest number of payment records
payments_per_order.head(10)

,order_id,payment_count
0,fa65dad1b0e818e3ccc5cb0e39231352,29
1,ccf804e764ed5650cd8759557269dc13,26
2,285c2e15bebd4ac83635ccc563dc71f4,22
3,895ab968e7bb0d5659d16cd74cd1650c,21
4,ee9ca989fc93ba09a6eddc250ce01742,19
5,fedcd9f7ccdc8cba3a18defedd1a5547,19
6,4bfcba9e084f46c8e3cb49b0fa6e6159,15
7,21577126c19bf11a0b91592e5844ba78,15
8,3c58bffb70dcf45f12bdf66a3c215905,14
9,4689b1816de42507a7d63a4617383c59,14


## 11. Check the Review Table Grain

I inspect the review table to determine how review records relate to orders.

This helps identify whether the review data also needs to be aggregated before joining it with the order-level data.

In [15]:
# Check the number of review records and unique orders
query = """
SELECT
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT order_id) AS orders_with_reviews
FROM order_reviews;
"""

reviews_check = pd.read_sql(query, engine)

print(reviews_check)

   total_reviews  orders_with_reviews
0          99224                98673


In [16]:
# Count review records for each order
query = """
SELECT
    order_id,
    COUNT(*) AS review_count
FROM order_reviews
GROUP BY order_id
ORDER BY review_count DESC;
"""

reviews_per_order = pd.read_sql(query, engine)

# Show orders with the highest number of reviews
reviews_per_order.head(10)

,order_id,review_count
0,03c939fd7fd3b38f8485a0f95798f1f6,3
1,8e17072ec97ce29f0e1f111e598b0c85,3
2,c88b1d1b157a9999ce368f218a407141,3
3,df56136b8031ecd28e200bb18e6ddb2e,3
4,bf9a3a8d9db6be5e4dfcfe2f071ef405,2
5,0dfedf3d432d2cdca59a3ef16b313db6,2
6,ce0102221c8d12a97979c74d09cca282,2
7,26ba6dc5d33b55a3107a56f4e8d77395,2
8,19fe6cd13dca5943f17abd2c37c46abd,2
9,3df55fc07ff463109ce0422439693aee,2


## 12. Inspect Product-Level Relationships

I inspect the relationship between order items and products.

The `order_items` table contains `product_id`, which can be used to connect each ordered item to product information.

In [17]:
# Check whether product_id is unique in the products table
query = """
SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT product_id) AS unique_products
FROM products;
"""

products_check = pd.read_sql(query, engine)

print(products_check)

   total_products  unique_products
0           32951            32951


In [18]:
# Check how many ordered items have a matching product
query = """
SELECT
    COUNT(*) AS total_items,
    COUNT(product_id) AS items_with_product_id,
    COUNT(DISTINCT product_id) AS products_in_orders
FROM order_items;
"""

item_product_check = pd.read_sql(query, engine)

print(item_product_check)

   total_items  items_with_product_id  products_in_orders
0       112650                 112650               32951


## 13. Inspect the Customer-Level Relationship

I inspect the relationship between orders and customers.

The `orders` table contains `customer_id`, which links each order to the corresponding customer.

In [19]:
# Check whether customer_id is unique in the customers table
query = """
SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM customers;
"""

customers_check = pd.read_sql(query, engine)

print(customers_check)

   total_customers  unique_customers
0            99441             99441


In [20]:
# Check customer IDs in the orders table
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(customer_id) AS orders_with_customer_id,
    COUNT(DISTINCT customer_id) AS unique_customers_in_orders
FROM orders;
"""

order_customer_check = pd.read_sql(query, engine)

print(order_customer_check)

   total_orders  orders_with_customer_id  unique_customers_in_orders
0         99441                    99441                       99441


## 14. Inspect Table Columns

I inspect the columns of each Olist table before loading the data.

This helps identify the keys and the fields needed for aggregation, joining, feature creation, and the final order-level dataset.

In [21]:
# Get the column names for each table

for table_name in tables["table_name"]:
    query = f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position;
    """

    columns = pd.read_sql(query, engine)

    print(f"\n{table_name}:")
    print(columns["column_name"].tolist())


customers:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

geolocation:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

order_items:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

order_payments:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

order_reviews:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

orders:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

product_category_translation:
['product_category_name', 'product_category_name_english']

products:
['product_id', 'product_category_name', 'product_name_lenght', 'product_d

## 15. Load the Main Olist Tables

I load the main tables needed for the order-level analysis into Pandas DataFrames.

I start with the tables directly related to orders and avoid loading unnecessary data until it is needed.

In [22]:
# Load the main tables from PostgreSQL

orders = pd.read_sql("SELECT * FROM orders", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments", engine)
order_reviews = pd.read_sql("SELECT * FROM order_reviews", engine)

In [23]:
# Check the number of rows loaded into each DataFrame

print("orders:", len(orders))
print("customers:", len(customers))
print("order_items:", len(order_items))
print("order_payments:", len(order_payments))
print("order_reviews:", len(order_reviews))

orders: 99441
customers: 99441
order_items: 112650
order_payments: 103886
order_reviews: 99224


## 16. Validate the Loaded Data

I verify the number of rows loaded into each DataFrame and compare them with the database row counts.

This confirms that the data was transferred correctly from PostgreSQL to Python.

In [24]:
# Check the number of rows in each DataFrame

print("orders:", len(orders))
print("customers:", len(customers))
print("order_items:", len(order_items))
print("order_payments:", len(order_payments))
print("order_reviews:", len(order_reviews))

orders: 99441
customers: 99441
order_items: 112650
order_payments: 103886
order_reviews: 99224


## 17. Check Missing Values in Key Columns

I check for missing values in the key columns that will be used for joining and aggregation.

This helps identify potential data quality issues before combining the tables.

In [25]:
# Check missing values in important key columns

key_columns = {
    "orders": ["order_id", "customer_id"],
    "customers": ["customer_id"],
    "order_items": ["order_id", "product_id"],
    "order_payments": ["order_id"],
    "order_reviews": ["order_id"]
}

for table_name, columns in key_columns.items():
    print(f"\n{table_name}:")

    df = globals()[table_name]

    for column in columns:
        missing = df[column].isna().sum()
        print(f"{column}: {missing} missing values")


orders:
order_id: 0 missing values
customer_id: 0 missing values

customers:
customer_id: 0 missing values

order_items:
order_id: 0 missing values
product_id: 0 missing values

order_payments:
order_id: 0 missing values

order_reviews:
order_id: 0 missing values


## 18. Aggregate Order Items

The `order_items` table contains multiple rows for some orders.

To keep one row per order, I aggregate the item-level data into order-level features:

- `item_count`: number of items in the order.
- `total_price`: total product price.
- `total_freight`: total freight value.

In [26]:
# Aggregate item-level data to one row per order

items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

# Check the result
items_agg.head()

,order_id,item_count,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


## 19. Aggregate Order Payments

The `order_payments` table can contain multiple payment records for the same order.

I aggregate the payment data to the order level so that the final dataset keeps one row per order.

In [27]:
# Aggregate payment data to one row per order

payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_agg.head()

,order_id,payment_count,total_payment,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


## 20. Aggregate Order Reviews

The `order_reviews` table may contain multiple review records for the same order.

I aggregate the review data to the order level so that it can be joined without creating duplicate order rows.

In [28]:
# Aggregate review data to one row per order

reviews_agg = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

reviews_agg.head()

,order_id,review_count,average_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,4.0
2,000229ec398224ef6ca0657da4fc703e,1,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0


## 21. Prepare the Order-Level Dataset

The `orders` table is used as the base table because each `order_id` is unique.

I then add the aggregated item, payment, and review features to create an order-level dataset.

In [29]:
# Start with the order-level table
final_df = orders.copy()

# Add aggregated item features
final_df = final_df.merge(
    items_agg,
    on="order_id",
    how="left"
)

# Add aggregated payment features
final_df = final_df.merge(
    payments_agg,
    on="order_id",
    how="left"
)

# Add aggregated review features
final_df = final_df.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

final_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_price,total_freight,payment_count,total_payment,payment_installments,review_count,average_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,3.0,38.71,1.0,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,1.0,141.46,1.0,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,1.0,179.12,3.0,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,1.0,72.20,1.0,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,1.0,28.62,1.0,1.0,5.0


## 22. Add Customer Information

I add customer information to the order-level dataset using `customer_id`.

This adds customer-level attributes while keeping the order as the main unit of analysis.

In [30]:
# Select customer information
customer_features = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
]

# Add customer information
final_df = final_df.merge(
    customer_features,
    on="customer_id",
    how="left"
)

## Add Seller Information

Seller information is added to the order-level dataset to support geographic analysis.

In [31]:
# Load sellers table from PostgreSQL

sellers = pd.read_sql(
    "SELECT * FROM sellers",
    engine
)

In [32]:
# Select seller information
seller_features = sellers[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
]

# Add seller information to order items
items_sellers = order_items.merge(
    seller_features,
    on="seller_id",
    how="left"
)

# Aggregate seller information to the order level
seller_agg = (
    items_sellers
    .groupby("order_id")
    .agg(
        seller_zip_code_prefix=("seller_zip_code_prefix", "first"),
        seller_city=("seller_city", "first"),
        seller_state=("seller_state", "first")
    )
    .reset_index()
)

# Add seller information to the final dataset
final_df = final_df.merge(
    seller_agg,
    on="order_id",
    how="left"
)

In [33]:
print("Final rows:", len(final_df))
print("Unique orders:", final_df["order_id"].nunique())
print("Seller state missing:", final_df["seller_state"].isna().sum())

Final rows: 99441
Unique orders: 99441
Seller state missing: 775


## 23. Load the Products Table

I load the products table from PostgreSQL because product attributes are needed to create order-level product features.

In [34]:
# Load the products table from PostgreSQL
products = pd.read_sql(
    "SELECT * FROM products",
    engine
)

print("Products loaded:", len(products))

Products loaded: 32951


## 24. Aggregate Product Information

Product information is linked through `order_items`, where an order can contain multiple products.

I aggregate selected product attributes to the order level before joining them with the final dataset.

In [35]:
# Add product information to each order item
items_products = order_items.merge(
    products[
        [
            "product_id",
            "product_weight_g",
            "product_photos_qty"
        ]
    ],
    on="product_id",
    how="left"
)

# Aggregate product information to the order level
product_agg = (
    items_products
    .groupby("order_id")
    .agg(
        unique_products=("product_id", "nunique"),
        average_product_weight=("product_weight_g", "mean"),
        average_product_photos=("product_photos_qty", "mean")
    )
    .reset_index()
)

product_agg.head()

,order_id,unique_products,average_product_weight,average_product_photos
0,00010242fe8c5a6d1ba2dd792cb16214,1,650.0,4.0
1,00018f77f2f0320c557190d7a144bdd3,1,30000.0,2.0
2,000229ec398224ef6ca0657da4fc703e,1,3050.0,2.0
3,00024acbcdf0a6daa1e931b038114c75,1,200.0,1.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,3750.0,1.0


In [36]:
# Add product features to the final dataset

final_df = final_df.merge(
    product_agg,
    on="order_id",
    how="left"
)

final_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_price,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,unique_products,average_product_weight,average_product_photos
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,...,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,09350,maua,SP,1.0,500.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,...,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,31570,belo horizonte,SP,1.0,400.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,...,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,14840,guariba,SP,1.0,420.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,...,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,31842,belo horizonte,MG,1.0,450.0,3.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,...,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,08752,mogi das cruzes,SP,1.0,250.0,4.0


## 24. Create Delivery Time Feature

I calculate the delivery time in days using the purchase and delivery timestamps.

This provides an order-level feature that describes how long each order took to be delivered.

In [37]:
# Convert order date columns to datetime
final_df["order_purchase_timestamp"] = pd.to_datetime(
    final_df["order_purchase_timestamp"]
)

final_df["order_delivered_customer_date"] = pd.to_datetime(
    final_df["order_delivered_customer_date"]
)

# Calculate delivery time in days
final_df["delivery_days"] = (
    final_df["order_delivered_customer_date"]
    - final_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

final_df[["order_id", "delivery_days"]].head()

,order_id,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877


## 25. Final Dataset Validation

I validate the final dataset to confirm that the joins preserved the required order-level grain.

The final dataset should contain one row per order.

In [38]:
# Check the final dataset size and order uniqueness

print("Final rows:", len(final_df))
print("Unique orders:", final_df["order_id"].nunique())

Final rows: 99441
Unique orders: 99441


## 26. Save the Final Dataset

I save the validated order-level dataset as a CSV file.

The dataset contains one row per order and includes the aggregated item, payment, review, customer, product, and delivery features created during the preparation process.

In [39]:
# Save the final order-level dataset

output_path = "../artifacts/final_order_dataset.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully.")
print("File:", output_path)

Final dataset saved successfully.
File: ../artifacts/final_order_dataset.csv


In [40]:
import os

# Check that the output file exists
print("File exists:", os.path.exists(output_path))

File exists: True
